# Phase 5 - Data Enrichment via Public API

**ITS 2122 - Python for Data Science & AI | Project: Credit Risk & Customer Financial Health Analytics**

This phase enriches the TWD-denominated portfolio with a current TWD-to-USD reporting conversion. It demonstrates public API collection with `requests`, transparent validation, and safe failure handling.

## Objectives

1. Retrieve a public TWD exchange-rate snapshot without a private API key.
2. Validate the HTTP response, JSON payload, service status, and required USD rate.
3. Create an API-enriched in-memory view with USD credit-limit and outstanding-exposure fields.
4. Explain how the conversion helps international portfolio reporting without changing risk decisions.

## API Choice and Business Relevance

The notebook uses the [ExchangeRate-API Open Access endpoint](https://www.exchangerate-api.com/docs/free): `https://open.er-api.com/v6/latest/TWD`. It is a public, no-key endpoint that returns daily rates for a TWD base currency; ExchangeRate-API attribution is retained below.

The project data are expressed in TWD. Converting selected exposure measures to USD gives international stakeholders a familiar reporting view, making it easier to compare the scale of the portfolio with other markets. The conversion is for reporting only: customer risk segmentation and interventions remain based on observed repayment behaviour in the original data.

In [1]:
import json

import pandas as pd
import requests
from requests import exceptions as request_exceptions

CLEANED_PATH = 'cleaned_credit_data.csv'
BASE_CURRENCY = 'TWD'
REPORTING_CURRENCY = 'USD'
API_URL = f'https://open.er-api.com/v6/latest/{BASE_CURRENCY}'
REQUEST_TIMEOUT_SECONDS = 15

print(f'API endpoint: {API_URL}')
print(f'Reporting conversion: 1 {BASE_CURRENCY} -> {REPORTING_CURRENCY}')

API endpoint: https://open.er-api.com/v6/latest/TWD
Reporting conversion: 1 TWD -> USD


## Error-Handling Policy

The API call is allowed to fail without breaking the notebook. The helper below handles timeout and connection failures, HTTP status errors, invalid JSON, unsuccessful service responses, and a missing or invalid USD rate. On failure it reports the reason and skips conversion rather than substituting an invented or stale rate.

In [2]:
def fetch_twd_to_usd_rate(url: str, timeout_seconds: int):
    """Fetch and validate one current TWD-to-USD reporting rate.

    Returns ``(snapshot, None)`` on success, or ``(None, message)``
    on failure. The caller can therefore skip conversion safely.
    """
    try:
        response = requests.get(url, timeout=timeout_seconds)
        response.raise_for_status()
    except request_exceptions.Timeout:
        return None, 'API request timed out; USD conversion was skipped.'
    except request_exceptions.ConnectionError:
        return None, 'Could not connect to the API; USD conversion was skipped.'
    except request_exceptions.HTTPError as error:
        return None, f'API returned HTTP {error.response.status_code}; USD conversion was skipped.'
    except request_exceptions.RequestException as error:
        return None, f'Unexpected API request error: {error}; USD conversion was skipped.'

    try:
        payload = response.json()
    except ValueError:
        return None, 'API response was not valid JSON; USD conversion was skipped.'

    if payload.get('result') != 'success':
        return None, f"API service response was not successful: {payload.get('error-type', 'unknown error')}."

    rate = payload.get('rates', {}).get(REPORTING_CURRENCY)
    if not isinstance(rate, (int, float)) or rate <= 0:
        return None, f"API response did not contain a valid {REPORTING_CURRENCY} rate."

    snapshot = {
        'base_code': payload.get('base_code'),
        'reporting_currency': REPORTING_CURRENCY,
        'conversion_rate': float(rate),
        'time_last_update_utc': payload.get('time_last_update_utc'),
        'time_next_update_utc': payload.get('time_next_update_utc'),
        'documentation': payload.get('documentation'),
    }
    return snapshot, None

In [3]:
api_snapshot, api_error = fetch_twd_to_usd_rate(
    API_URL,
    timeout_seconds=REQUEST_TIMEOUT_SECONDS,
)

if api_error:
    print(f'API integration warning: {api_error}')
else:
    # A compact sample keeps the notebook readable while showing real JSON fields.
    sample_json = {
        'result': 'success',
        'base_code': api_snapshot['base_code'],
        'time_last_update_utc': api_snapshot['time_last_update_utc'],
        'rates': {REPORTING_CURRENCY: api_snapshot['conversion_rate']},
        'documentation': api_snapshot['documentation'],
    }
    print('Validated API JSON sample:')
    print(json.dumps(sample_json, indent=2))
    print(
        f"\nConversion rate: 1 {BASE_CURRENCY} = "
        f"{api_snapshot['conversion_rate']:.6f} {REPORTING_CURRENCY}"
    )

Validated API JSON sample:
{
  "result": "success",
  "base_code": "TWD",
  "time_last_update_utc": "Mon, 13 Jul 2026 00:02:31 +0000",
  "rates": {
    "USD": 0.031181
  },
  "documentation": "https://www.exchangerate-api.com/docs/free"
}

Conversion rate: 1 TWD = 0.031181 USD


## Dataset Integration

The cleaned CSV is loaded unchanged. The notebook creates `credit_limit_usd` and `outstanding_exposure_usd` only in `api_enriched_df`, an in-memory reporting view. Outstanding exposure is the non-negative average bill, consistent with Phase 4: a negative bill is a customer credit balance, not lender exposure.

In [4]:
df = pd.read_csv(CLEANED_PATH)
assert df.shape == (30_000, 34), f'Unexpected cleaned-data shape: {df.shape}'

api_enriched_df = df.copy()
api_enriched_df['outstanding_exposure_twd'] = api_enriched_df['avg_bill_amount'].clip(lower=0)

if api_error:
    print('No USD columns or conversion summary were created because the API rate was unavailable.')
else:
    twd_to_usd = api_snapshot['conversion_rate']
    api_enriched_df['credit_limit_usd'] = api_enriched_df['credit_limit'] * twd_to_usd
    api_enriched_df['outstanding_exposure_usd'] = (
        api_enriched_df['outstanding_exposure_twd'] * twd_to_usd
    )

    conversion_summary = pd.DataFrame({
        'metric': [
            'Customers',
            'Portfolio default rate',
            'Total credit limit',
            'Total outstanding exposure',
        ],
        'value_twd': [
            None,
            None,
            api_enriched_df['credit_limit'].sum(),
            api_enriched_df['outstanding_exposure_twd'].sum(),
        ],
        'value_usd': [
            None,
            None,
            api_enriched_df['credit_limit_usd'].sum(),
            api_enriched_df['outstanding_exposure_usd'].sum(),
        ],
        'non_currency_value': [
            len(api_enriched_df),
            api_enriched_df['default_next_month'].mean() * 100,
            None,
            None,
        ],
    })

    print(
        f"Rate timestamp: {api_snapshot['time_last_update_utc']} | "
        f"1 TWD = {twd_to_usd:.6f} USD"
    )
    display(conversion_summary.round({
        'value_twd': 2, 'value_usd': 2, 'non_currency_value': 2,
    }))
    display(api_enriched_df[[
        'customer_id', 'credit_limit', 'credit_limit_usd',
        'outstanding_exposure_twd', 'outstanding_exposure_usd',
    ]].head())

Rate timestamp: Mon, 13 Jul 2026 00:02:31 +0000 | 1 TWD = 0.031181 USD


,metric,value_twd,value_usd,non_currency_value
0,Customers,NaN,NaN,30000.00
1,Portfolio default rate,NaN,NaN,22.12
2,Total credit limit,5.024530e+09,1.566699e+08,NaN
3,Total outstanding exposure,1.349654e+09,4.208357e+07,NaN


,customer_id,credit_limit,credit_limit_usd,outstanding_exposure_twd,outstanding_exposure_usd
0,1,20000,623.62,1284.000000,40.036404
1,2,120000,3741.72,2846.166667,88.746323
2,3,90000,2806.29,16942.166667,528.273699
3,4,50000,1559.05,38555.666667,1202.204242
4,5,50000,1559.05,18223.166667,568.216560


## Business Value and Interpretation

This enrichment lets an international executive board view TWD credit limits and outstanding exposure in USD at a clearly dated rate. It supports portfolio-scale comparison and reporting, while preserving TWD as the operational currency used in the historical account data. The rate must not be used as a customer-level risk score, pricing rule, or automated adverse-action input.

The converted values are a point-in-time reporting view. They do not measure realised foreign-exchange gains or losses, and they should be refreshed and relabelled whenever the notebook is rerun.

## API Limitations and Attribution

- The Open Access endpoint supplies a current daily rate, not a historical rate matched to each account observation month.
- API availability and rates can change between notebook runs; the displayed timestamp is therefore part of the evidence.
- If the API fails, the notebook reports the problem and deliberately produces no USD conversion rather than fabricating a fallback value.
- Exchange-rate data provided by [ExchangeRate-API](https://www.exchangerate-api.com/). No private API key is used or stored.

### Final-Appendix Evidence

In the merged notebook, keep the endpoint, the compact JSON sample, the displayed API timestamp, the conversion summary, and the failure-handling message visible. This is the evidence that the API requirement was met; do not hard-code a rate or present a previous run's value as current.

This temporary Phase 5 notebook will be incorporated into the final technical appendix after the remaining approved tasks are complete.